# 10 KV_Cache_and_Inference

## Problem

为什么大模型逐 token 生成时需要 KV cache？如果没有 cache，到底重复计算了什么？

## Dependencies

建议已经理解 self-attention 与 decoder-only 生成方式。


## Module_Goals

- 理解自回归生成的逐步展开过程
- 理解 K/V 为什么适合缓存，Q 为什么通常不缓存
- 理解有无 cache 的重复计算差异
- 能写一个最小缓存流程并比较张量增长方式

## Scope_And_Approach

这个 notebook 会先用直觉和小例子说明问题，再给出最小实现。重点是把模块为什么存在、输入输出是什么、关键步骤如何衔接说明清楚。


## 先用生活化比喻

假设你在一边写文章一边回顾前文。

- 没有 cache：每写一个新词，你都把整篇文章从头再读一遍，再重新做完整理解。
- 有 cache：前面读过的部分先记住重点摘要；新词来了，只需要把它和历史摘要对一遍。

这当然不是 attention 的精确等价物，但很接近它的工程直觉：**历史 token 的 K/V 一旦算出来，后续生成新 token 时不需要每次都重算。**

于是推理速度和显存结构就会被 KV cache 深刻影响。


In [ ]:
import numpy as np

np.random.seed(21)
np.set_printoptions(precision=3, suppress=True)

hidden_dim = 4
W_k = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_v = np.random.randn(hidden_dim, hidden_dim) * 0.2
W_q = np.random.randn(hidden_dim, hidden_dim) * 0.2

sequence = [
    np.array([1.0, 0.0, 0.5, 0.2]),
    np.array([0.8, 0.1, 0.3, 0.0]),
    np.array([0.1, 1.0, 0.4, 0.6]),
]


In [ ]:
def no_cache_generate(seq):
    total_kv_computes = 0
    for t in range(1, len(seq) + 1):
        prefix = np.stack(seq[:t])
        K = prefix @ W_k
        V = prefix @ W_v
        Q = prefix[-1:] @ W_q
        total_kv_computes += len(prefix)
        print(f't={t}: prefix_len={len(prefix)}, K.shape={K.shape}, V.shape={V.shape}, Q.shape={Q.shape}')
    return total_kv_computes


def with_cache_generate(seq):
    k_cache = []
    v_cache = []
    total_new_kv_computes = 0
    for t, token in enumerate(seq, start=1):
        k_new = token @ W_k
        v_new = token @ W_v
        q_new = token @ W_q
        k_cache.append(k_new)
        v_cache.append(v_new)
        total_new_kv_computes += 1
        print(f't={t}: cache_len={len(k_cache)}, new_q_shape={q_new.shape}')
    return total_new_kv_computes

print('无 cache 的 K/V 计算次数 =', no_cache_generate(sequence))
print('有 cache 的新 K/V 计算次数 =', with_cache_generate(sequence))


## Trade_Offs_And_Risk_Points

- 把 KV cache 理解成缓存整个 attention 输出。常见做法缓存的是每层历史 K/V。
- 以为有了 cache 就没有代价。它节省重复计算，但会带来显存占用与带宽压力。
- 不区分训练和推理。KV cache 主要是推理阶段的关键工程结构。

## Checkpoints

- 把序列长度改大，粗略估算有无 cache 的重复计算差距如何增长。
- 思考为什么 Q 通常只为当前新 token 计算，而 K/V 要保存历史。
- 解释为什么长上下文场景下，KV cache 会变成主要瓶颈之一。

## Summary

KV cache 的意义在于把自回归生成里的重复劳动压到更少。也正因为 cache 变得越来越大，后面我们才会特别关心 MQA、GQA、MLA 这些减少缓存压力的设计。

## Next_Module

下一节进入 MQA 和 GQA。那是从注意力头结构上减少 KV 成本的第一层关键优化。
